# 03 · Join Sofascore + Capology — England Premier League 21/22

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2021/22 de Premier League inglesa**.

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_england_2122.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_england_2122.csv').copy()

print(f'Sofascore:  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:   {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore:  538 jugadores | 116 columnas
Capology:   562 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   brighton hove albion
   leeds united
   leicester city
   newcastle united
   norwich city
   tottenham hotspur
   west ham united

En Capology pero no en Sofascore:
   brighton
   leeds
   leicester
   newcastle
   norwich
   tottenham
   west ham


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [6]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'brighton':'brighton hove albion',
            'leeds':'leeds united',
            'leicester':'leicester city',
            'newcastle':'newcastle united',
            'norwich':'norwich city',
            'tottenham':'tottenham hotspur',
            'west ham':'west ham united'
}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')


✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [7]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 480/538 (89.2%)
Sin emparejar: 58


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [8]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          8
Revisión media    (0.75 ≤ score < 0.90):   4
Revisión estricta (0.50 ≤ score < 0.75):   20
Revisión muy est. (score < 0.50):           26


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [9]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
0,Pierre-Emile Højbjerg,Tottenham Hotspur,pierre emile hojbjerg,0.976
17,Jacob Lungi Sørensen,Norwich City,jacob lungi sorensen,0.974
14,Mads Bech Sørensen,Brentford,mads bech sorensen,0.971
2,Christian Nørgaard,Brentford,christian norgaard,0.971
4,Łukasz Fabiański,West Ham United,lukasz fabianski,0.968
44,Joshua Dasilva,Brentford,joshua da silva,0.966
24,Przemysław Płacheta,Norwich City,przemyslaw placheta,0.944
16,Andriy Yarmolenko,West Ham United,andrii yarmolenko,0.941


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [10]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
6,Valentino Livramento,Southampton,tino livramento,0.857
18,Jóhann Guðmundsson,Burnley,johann berg gudmundsson,0.850
7,Edward Nketiah,Arsenal,eddie nketiah,0.815
8,Tariqe Fosu-Henry,Brentford,tariqe fosu,0.786


In [11]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')


Aceptados: 4 | Excluidos: 0


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [12]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
25,Mads Roerslev,Brentford,mads roerslev rasmussen,0.722
5,Emerson Royal,Tottenham Hotspur,emerson,0.700
50,Moise Kean,Everton,michael keane,0.696
27,Mahmoud Trézéguet,Aston Villa,trezeguet,0.692
38,Alejandro Garnacho,Manchester United,jadon sancho,0.667
12,Troy Deeney,Watford,tom cleverley,0.583
11,Francisco Trincão,Wolverhampton,trincao,0.583
40,Cameron Archer,Aston Villa,calum chambers,0.571
46,Kaide Gordon,Liverpool,andrew robertson,0.571
15,Samir Caetano,Watford,samir,0.556


In [13]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['mads roerslev',
                    'emerson royal',
                    'mahmoud trezeguet',
                    'francisco trincao',
                    'samir caetano',
                    'thiago alcantara'

]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')


Aceptados del nivel bajo: 6


### 7.4 Revisión muy estricta (score < 0.50)

Por defecto ninguno se acepta. Añadir a `ACCEPT_VERY_LOW_FUZZY` los correctos.

In [14]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
52,Emerson Palmieri,Chelsea,christian pulisic,0.485
56,Nathan Young-Coombes,Brentford,saman ghoddos,0.485
22,Harrison Ashby,West Ham United,darren randolph,0.483
53,Wesley Moraes,Aston Villa,ashley young,0.480
29,Sonny Perkins,West Ham United,winston reid,0.480
26,Jonathan Rowe,Norwich City,grant hanley,0.480
3,Liam Delap,Manchester City,aymeric laporte,0.480
13,Hélder Costa,Leeds United,tyler roberts,0.480
28,Evan Ferguson,Brighton & Hove Albion,dan burn,0.476
10,Tim Iroegbunam,Aston Villa,emiliano buendia,0.467


In [15]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')


Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [16]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 498/538 (92.6%)
Sin salario:     40


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [17]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 40


,player,team,minutesPlayed,appearances,goals,assists
0,Reiss Nelson,Arsenal,19,1,0,0
1,Tim Iroegbunam,Aston Villa,95,3,0,0
2,Cameron Archer,Aston Villa,30,3,0,0
3,Jaden Philogene-Bidace,Aston Villa,11,1,0,0
4,Wesley Moraes,Aston Villa,1,1,0,0
5,Nathan Young-Coombes,Brentford,3,1,0,0
6,Jeremy Sarmiento,Brighton & Hove Albion,56,5,0,0
7,Evan Ferguson,Brighton & Hove Albion,22,1,0,0
8,Emerson Palmieri,Chelsea,4,1,0,0
9,Jesurun Rak-Sakyi,Crystal Palace,81,2,0,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [18]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  Arsenal  —  SF sin salario:


,player,minutesPlayed
0,Reiss Nelson,19


  CG plantilla completa:


,player,player_norm
0,Aaron Ramsdale,aaron ramsdale
1,Ainsley Maitland-Niles,ainsley maitland niles
2,Albert Sambi Lokonga,albert sambi lokonga
3,Alexandre Lacazette,alexandre lacazette
4,Arthur Okonkwo,arthur okonkwo
5,Ben White,ben white
6,Bernd Leno,bernd leno
7,Bukayo Saka,bukayo saka
8,Calum Chambers,calum chambers
9,Cédric Soares,cedric soares



  Aston Villa  —  SF sin salario:


...

,player,player_norm
0,Aaron Cresswell,aaron cresswell
1,Alex Kral,alex kral
2,Alphonse Areola,alphonse areola
3,Andrii Yarmolenko,andrii yarmolenko
4,Angelo Ogbonna,angelo ogbonna
5,Arthur Masuaku,arthur masuaku
6,Ben Johnson,ben johnson
7,Craig Dawson,craig dawson
8,Darren Randolph,darren randolph
9,David Martin,david martin



  Wolverhampton  —  SF sin salario:


,player,minutesPlayed
0,Chem Campbell,12
1,Morgan Gibbs-White,18


  CG plantilla completa:


,player,player_norm
0,Adama Traoré,adama traore
1,Bruno Jordão,bruno jordao
2,Chiquinho,chiquinho
3,Conor Coady,conor coady
4,Daniel Podence,daniel podence
5,Fábio Silva,fabio silva
6,Hee-chan Hwang,hee chan hwang
7,João Moutinho,joao moutinho
8,John Ruddy,john ruddy
9,Jonny Otto,jonny otto


In [19]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {

}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')


Matches manuales definidos: 0


In [20]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')


Tras matches manuales: 498/538 (92.6%)


In [21]:
pd.reset_option('display.max_rows')

## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

In [22]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_england_2122.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_england_2122.csv
   Jugadores totales:  538
   Con salario:        498
   Sin salario (NaN):  40
   Columnas:           121
